# BioLit Search Assistant — NeMo-Ready

In [ ]:
import pandas as pd; pd.read_csv('sample_pubmed_abstracts.tsv', sep='\t').head()


## 📸 Screenshot Generator

This section saves example visuals (a UMAP plot and a sample search results table) to PNG files.
You can embed these in your README, slides, or job application materials.


In [ ]:

import matplotlib.pyplot as plt

# 1. Save UMAP scatter to PNG
import umap
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import NearestNeighbors
import pandas as pd

# Reload data
pm = pd.read_csv("sample_pubmed_abstracts.tsv", sep="\t")
on = pd.read_csv("sample_mondo_terms.tsv", sep="\t")

def fuse_pubmed(r):
    return f"{r['title']}. {r['abstract']}"

def fuse_ontology(r):
    return f"{r['label']}. {r['definition']}"

corpus = [fuse_pubmed(r) for _, r in pm.iterrows()] + [fuse_ontology(r) for _, r in on.iterrows()]
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
vecs = model.encode(corpus, normalize_embeddings=True)

reducer = umap.UMAP(random_state=42)
emb2d = reducer.fit_transform(vecs)
plt.figure(figsize=(6,5))
plt.scatter(emb2d[:,0], emb2d[:,1], alpha=0.7)
plt.title("2D UMAP Projection of BioLit Corpus")
plt.xlabel("UMAP-1"); plt.ylabel("UMAP-2")
plt.savefig("umap_projection.png", dpi=150)
plt.close()

print("Saved UMAP projection to umap_projection.png")

# 2. Save sample search results to PNG (via dataframe to image)
import dataframe_image as dfi

from sklearn.neighbors import NearestNeighbors
nn = NearestNeighbors(metric="cosine").fit(vecs)

query = "novel Alzheimer’s drug target"
qv = model.encode([query], normalize_embeddings=True)
distances, indices = nn.kneighbors(qv, n_neighbors=5)
sims = 1.0 - distances[0]

rows = []
for rank, (i, sim) in enumerate(zip(indices[0], sims), start=1):
    rows.append({"rank": rank, "text": corpus[i][:60] + "...", "similarity": float(sim)})
import pandas as pd
df_out = pd.DataFrame(rows)
dfi.export(df_out, "sample_search_results.png")

print("Saved sample search results to sample_search_results.png")
